In [ ]:
#Question 1: Image Preprocessing for Inference (PyTorch)
#Problem: Write a function to load an image and preprocess it for inference

In [ ]:
import torch
from torchvision import transforms
from PIL import Image

def preprocess_image_for_inference(image_path: str, target_size: int = 224) -> torch.Tensor:
    
    # 1. Load the image and force it into RGB format (drops alpha channels or handling grayscale)
    img = Image.open(image_path).convert('RGB')
    
    # 2. Define the exact ImageNet standard evaluation transformations
    preprocess_pipeline = transforms.Compose([
        # Resize the shorter side to target_size + 32 to preserve aspect ratio before cropping
        transforms.Resize(target_size + 32),
        
        # Center crop the image to the exact square resolution expected by the model
        transforms.CenterCrop(target_size),
        
        # Converts PIL Image (0-255) to a FloatTensor (0.0-1.0) and swaps dimensions to (C, H, W)
        transforms.ToTensor(),
        
        # Normalize channels using ImageNet channel mean and standard deviation parameters
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # 3. Apply the transformation pipeline
    input_tensor = preprocess_pipeline(img)
    
    # 4. Add a batch dimension at index 0: changes shape from (3, H, W) to (1, 3, H, W)
    # PyTorch models expect batches of images, even for a single image inference
    input_batch = input_tensor.unsqueeze(0)
    
    return input_batch


In [ ]:
#Question 2: Predict on New Image with a Trained Model
#Problem: Perform prediction and get the class label.

In [ ]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# Load the trained model
model = load_model("my_model.keras")

# Load a new image
img_path = "new_image.jpg"

img = image.load_img(
    img_path,
    target_size=(224, 224)
)

# Convert image to array
img_array = image.img_to_array(img)

# Add batch dimension
img_array = np.expand_dims(img_array, axis=0)

# Normalize pixel values
img_array = img_array / 255.0

# Make prediction
prediction = model.predict(img_array)

# Get predicted class index
predicted_class = np.argmax(prediction, axis=1)[0]

print("Predicted class:", predicted_class)

In [ ]:
#Question 3: Build a CNN to classify CIFAR-10 images (PyTorch)
#Problem: Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%.

In [ ]:
# ============================================
# 1. Import Libraries
# ============================================

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms


# ============================================
# 2. Device
# ============================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


# ============================================
# 3. Data Preprocessing
# ============================================

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])


# ============================================
# 4. Load CIFAR-10 Dataset
# ============================================

train_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# ============================================
# 5. DataLoaders
# ============================================

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2
)


# ============================================
# 6. CIFAR-10 Classes
# ============================================

classes = (
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
)


# ============================================
# 7. Build CNN Model
# ============================================

class CNN(nn.Module):

    def __init__(self):
        super(CNN, self).__init__()

        self.features = nn.Sequential(

            # First Convolution Block
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(2, 2),

            # Second Convolution Block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(2, 2),

            # Third Convolution Block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm2d(128),
            nn.MaxPool2d(2, 2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(256, 10)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x


# Create model
model = CNN().to(device)

print(model)


# ============================================
# 8. Loss Function and Optimizer
# ============================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# ============================================
# 9. Train the Model
# ============================================

num_epochs = 15

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {running_loss / len(train_loader):.4f}"
    )


# ============================================
# 10. Evaluate Model
# ============================================

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

In [ ]:
#Question 4: Identify Overfitting from Training Logs and Solve It
#Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping. (use mnist dataset)

In [ ]:
# ============================================
# 1. Import libraries
# ============================================

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.callbacks import EarlyStopping


# ============================================
# 2. Load MNIST dataset
# ============================================

(X_train, y_train), (X_test, y_test) = mnist.load_data()

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


# ============================================
# 3. Normalize the images
# ============================================

X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0


# ============================================
# 4. Build the model
# ============================================

model = models.Sequential([

    layers.Flatten(input_shape=(28, 28)),

    layers.Dense(128, activation="relu"),

    # Dropout to reduce overfitting
    layers.Dropout(0.5),

    layers.Dense(64, activation="relu"),

    # Another Dropout layer
    layers.Dropout(0.3),

    layers.Dense(10, activation="softmax")
])


# ============================================
# 5. Compile the model
# ============================================

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


# ============================================
# 6. Early Stopping
# ============================================

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)


# ============================================
# 7. Train the model
# ============================================

history = model.fit(
    X_train,
    y_train,

    epochs=30,

    batch_size=128,

    validation_split=0.2,

    callbacks=[early_stopping]
)


# ============================================
# 8. Evaluate on test data
# ============================================

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test
)

print("Test Accuracy:", test_accuracy)

In [ ]:
'''Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)
Problem: Use VGG16 for binary classification with fine-tuning
Collect dataset from the below sites or any other,
Kaggle Datasets : https://www.kaggle.com/datasets
Google Dataset Search : https://datasetsearch.research.google.com
Papers with Code – Datasets :https://paperswithcode.com/datasets
Roboflow Universe : https://universe.roboflow.com
ImageNet :  https://image-net.org/'''

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.callbacks import EarlyStopping

# --------------------------------------------------
# 1. Parameters
# --------------------------------------------------

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

TRAIN_DIR = "cats_dogs/train"
VAL_DIR = "cats_dogs/validation"


# --------------------------------------------------
# 2. Load training dataset
# --------------------------------------------------

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=True
)


# --------------------------------------------------
# 3. Load validation dataset
# --------------------------------------------------

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
    shuffle=False
)

print("Classes:", train_ds.class_names)


# --------------------------------------------------
# 4. Improve data loading performance
# --------------------------------------------------

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)


# --------------------------------------------------
# 5. Data augmentation
# --------------------------------------------------

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])


# --------------------------------------------------
# 6. Load pretrained VGG16
# --------------------------------------------------

base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)


# --------------------------------------------------
# 7. Freeze VGG16 layers
# --------------------------------------------------

base_model.trainable = False


# --------------------------------------------------
# 8. Build classification model
# --------------------------------------------------

inputs = layers.Input(shape=(224, 224, 3))

x = data_augmentation(inputs)

# VGG16 preprocessing
x = preprocess_input(x)

# Pretrained VGG16 feature extractor
x = base_model(x, training=False)

# Reduce feature maps
x = layers.GlobalAveragePooling2D()(x)

# New classification layers
x = layers.Dense(128, activation="relu")(x)

x = layers.Dropout(0.5)(x)

# Binary classification
outputs = layers.Dense(1, activation="sigmoid")(x)


model = models.Model(inputs, outputs)


# --------------------------------------------------
# 9. Compile model
# --------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# --------------------------------------------------
# 10. Display model architecture
# --------------------------------------------------

model.summary()


# --------------------------------------------------
# 11. Train only the new classification layers
# --------------------------------------------------

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)


# --------------------------------------------------
# 12. Evaluate model
# --------------------------------------------------

loss, accuracy = model.evaluate(val_ds)

print("Validation Loss:", loss)
print("Validation Accuracy:", accuracy)


# ==================================================
# 13. FINE-TUNING
# ==================================================

print("\nStarting Fine-Tuning...")


# Unfreeze VGG16
base_model.trainable = True


# Freeze all layers before block5_conv1
set_trainable = False

for layer in base_model.layers:

    if layer.name == "block5_conv1":
        set_trainable = True

    layer.trainable = set_trainable


# --------------------------------------------------
# 14. Recompile with very small learning rate
# --------------------------------------------------

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.00001
    ),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


# --------------------------------------------------
# 15. Fine-tune the model
# --------------------------------------------------

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[early_stopping]
)


# --------------------------------------------------
# 16. Final evaluation
# --------------------------------------------------

loss, accuracy = model.evaluate(val_ds)

print("\nFinal Validation Loss:", loss)
print("Final Validation Accuracy:", accuracy)


# --------------------------------------------------
# 17. Save the model
# --------------------------------------------------

model.save("vgg16_cats_dogs_finetuned.keras")

print("\nModel saved successfully!")